# Breast cohort MC simulations and graph ablations

This notebook isolates the real breast cohort experiments. It separates two layers that should not be conflated: standalone MC simulation experiments and MC evaluation of trained graph-ablation variants.

## Experimental layers

- Standalone MC experiment lineage: uncertainty/calibration evaluations run on the original consistent rollout and endpoint-calibrated model variants.
- Graph ablations: trained-model controls that test whether scheduled sampling and graph message-passing edges matter after endpoint calibration.
- MC evaluation of ablations: the same residual MC wrapper applied to the ablated trained models.

## Standalone MC experiment lineage, T0 to T3

| label                            |   n_patients |   det_ftv_mae_ml |   mc_mean_ftv_mae_ml |   crps_ftv_ml |   raw_coverage90 |   conformal_coverage90 |   conformal_width90_ml |   swd_mm_mc |   chamfer_mm_mc |   alive_abs_err_mc |
|:---------------------------------|-------------:|-----------------:|---------------------:|--------------:|-----------------:|-----------------------:|-----------------------:|------------:|----------------:|-------------------:|
| Original consistent rollout MC   |          758 |           12.878 |               15.262 |         8.354 |            0.766 |                  0.901 |                 59.515 |       1.32  |           3.656 |             40.783 |
| Endpoint FTV 0.10, no alive mass |          758 |            5.995 |                7.306 |         5.145 |            0.875 |                  0.901 |                 26.923 |       1.214 |           3.498 |              7.005 |
| Endpoint FTV 0.10 + light alive  |          758 |            5.987 |                7.525 |         5.186 |            0.883 |                  0.901 |                 29.138 |       1.205 |           3.485 |              2.704 |
| Endpoint FTV 0.20 + alive 0.05   |          758 |            5.881 |                7.609 |         5.156 |            0.876 |                  0.901 |                 27.949 |       1.207 |           3.485 |              2.692 |

The major gain comes from endpoint-calibrated training, not from the MC wrapper alone. Relative to the original consistent rollout MC, endpoint calibration sharply lowers deterministic FTV MAE, CRPS, interval width, and alive-count error while keeping conformal coverage at the nominal level.

![Breast MC lineage endpoint](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_mc_lineage_t0_t3_endpoint.png)

![Breast MC lineage coverage](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_mc_lineage_t0_t3_coverage_width.png)

## MC evaluation of graph ablations, T0 to T3

| label                 |   n_patients |   det_ftv_mae_ml |   mc_mean_ftv_mae_ml |   crps_ftv_ml |   raw_coverage90 |   conformal_coverage90 |   conformal_width90_ml |   swd_mm_mc |   chamfer_mm_mc |   alive_abs_err_mc |
|:----------------------|-------------:|-----------------:|---------------------:|--------------:|-----------------:|-----------------------:|-----------------------:|------------:|----------------:|-------------------:|
| Full endpoint model   |          758 |            5.881 |                7.609 |         5.156 |            0.876 |                  0.901 |                 27.949 |       1.207 |           3.485 |              2.692 |
| No scheduled sampling |          758 |            5.881 |                7.609 |         5.156 |            0.876 |                  0.901 |                 27.949 |       1.207 |           3.485 |              2.692 |
| No graph edges        |          758 |            5.28  |                6.541 |         4.525 |            0.894 |                  0.901 |                 23.93  |       1.211 |           3.493 |              2.654 |

The no-scheduled-sampling row is effectively identical to the full endpoint model because the selected full-model checkpoint occurred before scheduled sampling became active. Therefore, that row should be treated as a null comparison, not as evidence that scheduled sampling never matters.

The no-edge model improves scalar endpoint metrics and interval width while leaving spatial MC metrics nearly unchanged. That suggests the current hand-defined message-passing edges are not helping scalar FTV after endpoint calibration. It does not remove the value of the graph-state representation, which is still what permits node-level alive state, spatial clouds, and tumor-state outputs.

![Breast ablation endpoint](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_ablation_t0_t3_endpoint.png)

![Breast ablation coverage](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_ablation_t0_t3_coverage_width.png)

## Conditioning-to-T3 behavior: standalone MC experiments

| label                            | cohort          |   n_patients |   mc_mean_ftv_mae_ml |   crps_ftv_ml |   conformal_width90_ml |   conformal_coverage90 |
|:---------------------------------|:----------------|-------------:|---------------------:|--------------:|-----------------------:|-----------------------:|
| Original consistent rollout MC   | breast_T0_to_T3 |          758 |               15.262 |         8.354 |                 59.515 |                  0.901 |
| Original consistent rollout MC   | breast_T1_to_T3 |          758 |               14.875 |         8.098 |                 60.144 |                  0.901 |
| Original consistent rollout MC   | breast_T2_to_T3 |          758 |               13.802 |         7.49  |                 60.25  |                  0.901 |
| Endpoint FTV 0.10, no alive mass | breast_T0_to_T3 |          758 |                7.306 |         5.145 |                 26.923 |                  0.901 |
| Endpoint FTV 0.10, no alive mass | breast_T1_to_T3 |          758 |                7.17  |         4.875 |                 31.004 |                  0.901 |
| Endpoint FTV 0.10, no alive mass | breast_T2_to_T3 |          758 |                5.887 |         4.016 |                 24.559 |                  0.901 |
| Endpoint FTV 0.10 + light alive  | breast_T0_to_T3 |          758 |                7.525 |         5.186 |                 29.138 |                  0.901 |
| Endpoint FTV 0.10 + light alive  | breast_T1_to_T3 |          758 |                7.136 |         4.898 |                 30.448 |                  0.901 |
| Endpoint FTV 0.10 + light alive  | breast_T2_to_T3 |          758 |                5.806 |         4.029 |                 24.834 |                  0.901 |
| Endpoint FTV 0.20 + alive 0.05   | breast_T0_to_T3 |          758 |                7.609 |         5.156 |                 27.949 |                  0.901 |
| Endpoint FTV 0.20 + alive 0.05   | breast_T1_to_T3 |          758 |                7.185 |         4.889 |                 29.613 |                  0.901 |
| Endpoint FTV 0.20 + alive 0.05   | breast_T2_to_T3 |          758 |                5.802 |         4.025 |                 24.193 |                  0.901 |

![Breast MC lineage conditioning](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_mc_lineage_conditioning_t3.png)

## Conditioning-to-T3 behavior: graph ablations

| label                 | cohort          |   n_patients |   mc_mean_ftv_mae_ml |   crps_ftv_ml |   conformal_width90_ml |   conformal_coverage90 |
|:----------------------|:----------------|-------------:|---------------------:|--------------:|-----------------------:|-----------------------:|
| Full endpoint model   | breast_T0_to_T3 |          758 |                7.609 |         5.156 |                 27.949 |                  0.901 |
| Full endpoint model   | breast_T1_to_T3 |          758 |                7.185 |         4.889 |                 29.613 |                  0.901 |
| Full endpoint model   | breast_T2_to_T3 |          758 |                5.802 |         4.025 |                 24.193 |                  0.901 |
| No scheduled sampling | breast_T0_to_T3 |          758 |                7.609 |         5.156 |                 27.949 |                  0.901 |
| No scheduled sampling | breast_T1_to_T3 |          758 |                7.185 |         4.889 |                 29.613 |                  0.901 |
| No scheduled sampling | breast_T2_to_T3 |          758 |                5.802 |         4.025 |                 24.193 |                  0.901 |
| No graph edges        | breast_T0_to_T3 |          758 |                6.541 |         4.525 |                 23.93  |                  0.901 |
| No graph edges        | breast_T1_to_T3 |          758 |                6.396 |         4.392 |                 26.899 |                  0.901 |
| No graph edges        | breast_T2_to_T3 |          758 |                5.332 |         3.776 |                 22.372 |                  0.901 |

![Breast ablation conditioning](../../reports/bio_ftv_synthetic_ablation_analysis/figures/breast_ablation_conditioning_t3.png)